# 02 — Feature Exploration

Runs the linguistic + stylometry feature extractors and the baseline/drift engine, then visualizes whether the simulated insiders (synthetic dataset) or flagged users (real dataset) actually separate from everyone else in feature space — the core validation question for the whole approach.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.ingest import load_email_data, load_insider_labels
from src.data.anonymize import anonymize_dataframe, pseudonymize_user
from src.features.linguistic_features import extract_features_df
from src.features.stylometry import extract_stylometry_df
from src.features.baseline_engine import BaselineEngine
from src.features.drift_scoring import score_drift_df, aggregate_user_risk

sns.set_style('whitegrid')

## Run the feature pipeline

In [ ]:
raw = load_email_data()
anon = anonymize_dataframe(raw)
feats = extract_features_df(anon)
feats = extract_stylometry_df(feats)
feats.filter(regex='sentiment|urgency|readability|lexical|function_word').describe()

## Sentiment vs. urgency, colored by ground-truth label

In [ ]:
labels = load_insider_labels()
plot_df = feats.copy()
if labels is not None:
    labels = labels.copy()
    labels['user'] = labels['user'].map(pseudonymize_user)
    plot_df = plot_df.merge(labels, on='user', how='left')
    plot_df['is_malicious_insider'] = plot_df['is_malicious_insider'].fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=plot_df, x='sentiment_polarity', y='urgency_score',
                     hue='is_malicious_insider', palette={0: '#0f62fe', 1: '#da1e28'},
                     alpha=0.5, ax=ax)
    ax.set_title('Sentiment polarity vs. urgency score')
    plt.show()
else:
    print('No labels available for this plot.')

## Baseline drift: does the sliding-window z-score separate insiders?

In [ ]:
engine = BaselineEngine()
z_df = engine.replay(feats)
scored = score_drift_df(z_df)
user_risk = aggregate_user_risk(scored)

if labels is not None:
    user_risk_labeled = user_risk.merge(labels, on='user', how='left')
    user_risk_labeled['is_malicious_insider'] = user_risk_labeled['is_malicious_insider'].fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=user_risk_labeled, x='is_malicious_insider', y='avg_drift_score',
                hue='is_malicious_insider', palette={0: '#0f62fe', 1: '#da1e28'},
                legend=False, ax=ax)
    ax.set_xticklabels(['normal', 'insider'])
    ax.set_title('Average drift score: normal vs. simulated insider users')
    plt.show()
else:
    print('No labels available for this plot.')

## Drift score trend for a single flagged user

In [ ]:
top_user = user_risk.sort_values('avg_drift_score', ascending=False).iloc[0]['user']
user_trend = scored[scored['user'] == top_user].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(user_trend.index, user_trend['drift_score'], color='#da1e28')
ax.axhline(y=user_trend['drift_score'].mean(), color='#8d8d8d', linestyle='--', label='mean')
ax.set_title(f'Drift score over time — {top_user}')
ax.set_xlabel('message index (chronological)')
ax.legend()
plt.tight_layout()
plt.show()

## Takeaways

- If the boxplot above shows separation between normal and insider groups, that's evidence the linguistic-drift signal carries real information beyond noise — worth reporting explicitly in the writeup, including the caveat that this is validated against a *synthetic* insider simulation, not real incidents.
- The per-user trend line is the same thing `frontend/index.html`'s drift chart shows an analyst — this notebook cell is effectively a debug view of that chart.